## HBONDING BETWEEN RESIDUES

In [ ]:
import MDAnalysis as mda
from parmed import load_file
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
u = mda.Universe(load_file('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/'), '/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/5ms-traj.dcd')


In [ ]:
hbonds = HydrogenBondAnalysis(universe=u, between=['protein', 'protein'])

protein_hydrogens_sel = hbonds.guess_hydrogens("protein")
protein_acceptors_sel = hbonds.guess_acceptors("protein")

hbonds.hydrogens_sel = f"({protein_hydrogens_sel})"
hbonds.acceptors_sel = f"({protein_acceptors_sel})"


hbonds.run(start=4000,stop=-1,step=50000,verbose=True)

In [ ]:
df = pd.DataFrame(hbonds.count_by_type(), columns = ['res1','res2','count'])
df['count'] = df['count'].astype(int)
df['count'] = df['count'] / len(hbonds.frames)
df = df.sort_values(by='count', ascending=False)
df['res1'] = df['res1'].str[:3]
df['res2'] = df['res2'].str[:3]
df = df.groupby(['res1', 'res2']).agg({'count': ['sum']}).reset_index()
df.columns = df.columns.droplevel(1)
df = df.sort_values(by='count', ascending=False)
df.to_csv('/Users/williambrown/Desktop/final_figures/HBond_figs/125_1_hbond_btwn.csv')


In [ ]:
array = np.array(df)

res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
            'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

heatmap_df = pd.DataFrame(index=res_list, columns=res_list)

for i in res_list:
    for j in res_list:
        if df['res1'].str.contains(i).any():
            temp_df = df[df['res1'] == i]
            new_df = temp_df[temp_df['res2'] == j]
            if new_df.shape[0] != 0:
                value = float(new_df['count'])
            else:
                value = 0

            heatmap_df.loc[i,j] = value
            heatmap_df.loc[j,i] = value
            
#         elif df['res2'].str.contains(j).any():
#             temp_df = df[df['res2'] == j]
#             new_df = temp_df[temp_df['res1'] == i]
#             if new_df.shape[0] != 0:
#                 value = float(new_df['count'])
#             else:
#                 value = 0

#             heatmap_df.loc[i,j] = value
#             heatmap_df.loc[j,i] = value
            
heatmap_df = heatmap_df.fillna(0)
heatmap_df

In [ ]:
ax = sns.heatmap(heatmap_df, cmap='YlGnBu')
ax.invert_yaxis()
ax.set_ylabel('Residue')
ax.set_xlabel('Residue')
ax.set_title('Water Mediated Interactions - 125 mg/mL')
plt.tight_layout()
plt.savefig('/Users/williambrown/Desktop/final_figures/HBond_figs/hb_bridge_heatmap.png', dpi=300)